In [ ]:
# Hypertension Risk Prediction
# Jupyter-ready notebook
# Steps: EDA, preprocessing, Decision Tree (Entropy & Gini), simple Random Forest (bagging), plots, feature importance, and report export.

In [ ]:
# 0. Install required packages (uncomment if needed)
# !pip install scikit-learn matplotlib pandas reportlab scipy

In [ ]:
# 1. Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# Optional for saving a small PDF summary
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
from reportlab.lib import colors

In [ ]:
# 2. Load dataset
DATA_PATH = 'hypertension.csv'  # change to your file (upload to the same folder or provide path)

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded dataset from {DATA_PATH} — shape: {df.shape}")
else:
    print("No dataset found at 'hypertension.csv'. Generating a synthetic dataset for demonstration...")
    np.random.seed(42)
    n = 2000
    age = np.random.randint(18, 80, size=n)
    salt_intake = np.clip(np.random.normal(8, 3, size=n), 0, 30)
    stress_score = np.clip(np.random.normal(4, 2.2, size=n), 0, 10)
    bp_history = np.random.choice(['Normal','Prehypertension','Hypertension'], size=n, p=[0.6,0.25,0.15])
    sleep_duration = np.clip(np.random.normal(7, 1.2, size=n), 3, 12)
    bmi = np.clip(np.random.normal(24, 4.5, size=n), 15, 45)
    medication = np.random.choice(['None','Beta Blocker','Diuretic','ACE Inhibitor','Other'], size=n, p=[0.7,0.08,0.06,0.1,0.06])
    family_history = np.random.choice(['Yes','No'], size=n, p=[0.25,0.75])
    exercise_level = np.random.choice(['Low','Moderate','High'], size=n, p=[0.4,0.45,0.15])
    smoking_status = np.random.choice(['Smoker','Non-Smoker'], size=n, p=[0.22,0.78])

    logit = -5 + 0.05*(age) + 0.08*(salt_intake) + 0.25*(stress_score) + 0.12*(bmi)
    logit += np.where(bp_history=='Prehypertension', 1.0, 0.0)
    logit += np.where(bp_history=='Hypertension', 2.0, 0.0)
    logit += np.where(medication!='None', 0.8, 0.0)
    logit += np.where(family_history=='Yes', 0.6, 0.0)
    logit += np.where(exercise_level=='Low', 0.5, -0.5)
    logit += np.where(smoking_status=='Smoker', 0.4, 0.0)

    prob = 1 / (1 + np.exp(-logit))
    has_hypertension = np.where(np.random.rand(n) < prob, 'Yes', 'No')

    df = pd.DataFrame({
        'Age': age,
        'Salt_Intake': np.round(salt_intake,2),
        'Stress_Score': np.round(stress_score,2),
        'BP_History': bp_history,
        'Sleep_Duration': np.round(sleep_duration,2),
        'BMI': np.round(bmi,2),
        'Medication': medication,
        'Family_History': family_history,
        'Exercise_Level': exercise_level,
        'Smoking_Status': smoking_status,
        'Has_Hypertension': has_hypertension
    })
    for col in ['Salt_Intake','Stress_Score','Sleep_Duration','BMI','Exercise_Level']:
        idx = np.random.choice(df.index, size=int(0.03*len(df)), replace=False)
        df.loc[idx, col] = np.nan
    print(f"Synthetic dataset created — shape: {df.shape}")

In [ ]:
# 3. Quick EDA - basic stats and missing values
print('\n--- Dataset Head ---')
display(df.head())

print('\n--- Data types ---')
print(df.dtypes)

print('\n--- Missing values per column ---')
print(df.isnull().sum())

print('\n--- Basic statistics (numerical) ---')
print(df.describe().T)

In [ ]:
# 4. Visualizations (matplotlib) — at least 3 plots
FIG_DIR = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)

plt.figure(figsize=(8,4))
plt.hist(df['Age'].dropna(), bins=14)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,'age_hist.png'))
plt.show()

plt.figure(figsize=(8,4))
plt.boxplot([df[df['Has_Hypertension']=='No']['BMI'].dropna(), df[df['Has_Hypertension']=='Yes']['BMI'].dropna()], labels=['No','Yes'])
plt.title('BMI vs Hypertension (boxplot)')
plt.ylabel('BMI')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,'bmi_box.png'))
plt.show()

plt.figure(figsize=(8,4))
plt.hist(df[df['Has_Hypertension']=='No']['Salt_Intake'].dropna(), bins=20, alpha=0.7, label='No')
plt.hist(df[df['Has_Hypertension']=='Yes']['Salt_Intake'].dropna(), bins=20, alpha=0.5, label='Yes')
plt.title('Salt Intake Distribution by Hypertension Status')
plt.xlabel('Salt Intake (g/day)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,'salt_hist.png'))
plt.show()

num_cols = ['Age','Salt_Intake','Stress_Score','Sleep_Duration','BMI']
plt.figure(figsize=(6,5))
plt.matshow(df[num_cols].corr(), fignum=1)
plt.xticks(range(len(num_cols)), num_cols, rotation=45)
plt.yticks(range(len(num_cols)), num_cols)
plt.colorbar()
plt.title('Correlation Matrix (numerical features)', pad=20)
plt.savefig(os.path.join(FIG_DIR,'corr_matrix.png'))
plt.show()

In [ ]:
# 5. Preprocessing pipeline (impute, encode, scale)
categorical_cols = ['BP_History','Medication','Family_History','Exercise_Level','Smoking_Status']
numerical_cols = ['Age','Salt_Intake','Stress_Score','Sleep_Duration','BMI']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_cols),
    ('cat', cat_pipeline, categorical_cols)
])

X = df.drop(columns=['Has_Hypertension'])
y = df['Has_Hypertension'].map({'No':0,'Yes':1}).values

X_pre = preprocessor.fit_transform(X)
num_feat_names = numerical_cols
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
ohe_names = list(ohe.get_feature_names_out(categorical_cols))
feature_names = list(num_feat_names) + ohe_names
print('\nTotal features after preprocessing:', len(feature_names))

In [ ]:
# 6. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_pre, y, test_size=0.2, random_state=42, stratify=y)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)

In [ ]:
# 7. Decision Tree (Entropy) — sweep max_depth and record accuracies
max_depths = list(range(1,16))
train_acc = []
val_acc = []
models_entropy = {}

for d in max_depths:
    clf = DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, clf.predict(X_train)))
    val_acc.append(accuracy_score(y_test, clf.predict(X_test)))
    models_entropy[d] = clf

plt.figure(figsize=(8,4))
plt.plot(max_depths, train_acc, marker='o')
plt.plot(max_depths, val_acc, marker='o')
plt.title('Train & Validation Accuracy vs max_depth (Decision Tree - Entropy)')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.legend(['Train Accuracy','Validation Accuracy'])
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,'accuracy_vs_depth_entropy.png'))
plt.show()

best_depth = max_depths[int(np.argmax(val_acc))]
print('Best entropy max_depth:', best_depth)
best_clf_entropy = models_entropy[best_depth]

In [ ]:
# 8. Evaluate best Entropy tree
y_pred = best_clf_entropy.predict(X_test)
print('\nEntropy Decision Tree — Classification Report:')
print(classification_report(y_test, y_pred, target_names=['No','Yes']))

cm = confusion_matrix(y_test, y_pred)
print('\nConfusion Matrix:\n', cm)

plt.figure(figsize=(4,3))
plt.matshow(cm, fignum=1)
plt.title('Confusion Matrix (Entropy DT)')
plt.colorbar()
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks([0,1], ['No','Yes'])
plt.yticks([0,1], ['No','Yes'])
plt.savefig(os.path.join(FIG_DIR,'confusion_matrix_entropy.png'))
plt.show()

importances = best_clf_entropy.feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)
print('\nTop 10 features (Entropy DT):')
print(feat_imp.head(10))

In [ ]:
# 9. Decision Tree (Gini) sweep and evaluation
models_gini = {}
train_acc_g = []
val_acc_g = []
for d in max_depths:
    clf_g = DecisionTreeClassifier(criterion='gini', max_depth=d, random_state=42)
    clf_g.fit(X_train, y_train)
    train_acc_g.append(accuracy_score(y_train, clf_g.predict(X_train)))
    val_acc_g.append(accuracy_score(y_test, clf_g.predict(X_test)))
    models_gini[d] = clf_g

plt.figure(figsize=(8,4))
plt.plot(max_depths, train_acc_g, marker='o')
plt.plot(max_depths, val_acc_g, marker='o')
plt.title('Train & Validation Accuracy vs max_depth (Decision Tree - Gini)')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.legend(['Train Accuracy','Validation Accuracy'])
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,'accuracy_vs_depth_gini.png'))
plt.show()

best_depth_g = max_depths[int(np.argmax(val_acc_g))]
print('Best gini max_depth:', best_depth_g)
best_clf_gini = models_gini[best_depth_g]

print('\nGini Decision Tree — Classification Report:')
print(classification_report(y_test, best_clf_gini.predict(X_test), target_names=['No','Yes']))

In [ ]:
# 10. Simple Random Forest from scratch (bagging + random feature subset per tree)
from scipy.stats import mode
def simple_random_forest(X_train, y_train, X_test, n_estimators=25, max_depth=6, max_features=None, random_state=42):
    np.random.seed(random_state)
    n_samples, n_features = X_train.shape
    if max_features is None:
        max_features = int(np.sqrt(n_features))
    preds = np.zeros((n_estimators, X_test.shape[0]), dtype=int)
    trees = []
    for i in range(n_estimators):
        idxs = np.random.choice(n_samples, size=n_samples, replace=True)
        X_boot = X_train[idxs]
        y_boot = y_train[idxs]
        feat_idx = np.random.choice(n_features, size=max_features, replace=False)
        clf = DecisionTreeClassifier(criterion='gini', max_depth=max_depth, random_state=random_state+i)
        clf.fit(X_boot[:, feat_idx], y_boot)
        preds[i, :] = clf.predict(X_test[:, feat_idx])
        trees.append((clf, feat_idx))
    majority = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=preds)
    return majority, trees

rf_preds, rf_trees = simple_random_forest(X_train, y_train, X_test, n_estimators=35, max_depth=8, max_features=None)
rf_acc = accuracy_score(y_test, rf_preds)
print('\nSimple Random Forest accuracy:', rf_acc)
print('\nRandom Forest — Classification Report:')
print(classification_report(y_test, rf_preds, target_names=['No','Yes']))

cm_rf = confusion_matrix(y_test, rf_preds)
print('\nRF Confusion Matrix:\n', cm_rf)

In [ ]:
# 11. Save models and preprocessing objects (optional)
import pickle
os.makedirs('models', exist_ok=True)
with open('models/preprocessor.pkl','wb') as f:
    pickle.dump(preprocessor, f)
with open('models/best_entropy_tree.pkl','wb') as f:
    pickle.dump(best_clf_entropy, f)
with open('models/best_gini_tree.pkl','wb') as f:
    pickle.dump(best_clf_gini, f)
print('\nSaved preprocessing pipeline and models to ./models/')

In [ ]:
# 12. Optional: Generate a short PDF report (basic)
try:
    report_path = 'Hypertension_Dataset_Report.pdf'
    doc = SimpleDocTemplate(report_path, pagesize=A4, rightMargin=30,leftMargin=30, topMargin=30,bottomMargin=18)
    styles = getSampleStyleSheet()
    styleN = styles['BodyText']
    styleH = styles['Heading1']
    elements = []
    elements.append(Paragraph('Hypertension Risk Prediction — Summary', styleH))
    elements.append(Spacer(1,8))
    elements.append(Paragraph(f'Number of samples: {len(df)}', styleN))
    elements.append(Spacer(1,6))
    mv_data = [['Column','Missing Count']] + [[c, str(int(v))] for c,v in df.isnull().sum().items()]
    mv_table = Table(mv_data, colWidths=[200,80])
    mv_table.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.lightblue),('INNERGRID',(0,0),(-1,-1),0.25,colors.black)]))
    elements.append(mv_table)
    elements.append(Spacer(1,8))
    for fname in ['age_hist.png','bmi_box.png','salt_hist.png','accuracy_vs_depth_entropy.png','confusion_matrix_entropy.png']:
        p = os.path.join(FIG_DIR,fname)
        if os.path.exists(p):
            elements.append(Image(p, width=450, height=250))
            elements.append(Spacer(1,6))
    doc.build(elements)
    print('\nPDF report saved at', report_path)
except Exception as e:
    print('Could not create PDF report — error:', e)

In [ ]:
print('\nRecommended next steps:')
print('- Replace synthetic data with your real Kaggle CSV and rerun.')
print('- Use GridSearchCV / cross_val_score for robust hyperparameter tuning.')
print('- Consider SHAP for interpretability and XGBoost/LightGBM for stronger baselines.')
print('\nEnd of notebook.')